# Grass Pollen Prediction for Stockholm

This notebook demonstrates pollen prediction using historical data and weather features.

In [ ]:
import sys
from pathlib import Path
import warnings
warnings.filterwarnings("ignore", module="IPython")

def is_google_colab() -> bool:
    if "google.colab" in str(get_ipython()):
        return True
    return False

def clone_repository() -> None:
    !git clone https://github.com/featurestorebook/mlfs-book.git
    %cd mlfs-book

def install_dependencies() -> None:
    !pip install --upgrade uv
    !uv pip install --all-extras --system --requirement pyproject.toml

if is_google_colab():
    clone_repository()
    install_dependencies()
    root_dir = str(Path().absolute())
    print("Google Colab environment")
else:
    root_dir = Path().absolute()
    if root_dir.parts[-1:] == ('pollen',):
        root_dir = Path(*root_dir.parts[:-1])
    if root_dir.parts[-1:] == ('notebooks',):
        root_dir = Path(*root_dir.parts[:-1])
    root_dir = str(root_dir) 
    print("Local environment")

print(f"Root dir: {root_dir}")

# Add the root directory to the `PYTHONPATH` 
if root_dir not in sys.path:
    sys.path.append(root_dir)
    print(f"Added the following directory to the PYTHONPATH: {root_dir}")

# Set the environment variables from the file <root_dir>/.env
from mlfs import config
settings = config.HopsworksSettings(_env_file=f"{root_dir}/.env")

In [ ]:
import pandas as pd
import hopsworks
from mlfs.airquality import util
import json
import warnings
warnings.filterwarnings("ignore")

## 1. Connect to Hopsworks and Save Location as Secret

In [ ]:
# Login to Hopsworks
project = hopsworks.login(engine="python")

# Get secrets API
secrets = project.get_secrets_api()

# Stockholm location details
country = "Sweden"
city = "Stockholm"
street = "Österväg 17"

# Stockholm coordinates
latitude = 59.3293
longitude = 18.0686

dict_obj = {
    "country": country,
    "city": city,
    "street": street,
    "latitude": latitude,
    "longitude": longitude
}

# Convert the dictionary to a JSON string
str_dict = json.dumps(dict_obj)

# Replace any existing secret with the new value
secret = secrets.get_secret("SENSOR_LOCATION_JSON")
if secret is not None:
    secret.delete()
    print("Replacing existing SENSOR_LOCATION_JSON")

secrets.create_secret("SENSOR_LOCATION_JSON", str_dict)

print(f"Location saved: {street}, {city}, {country}")

## 2. Load Historical Data

In [ ]:


# Get historical pollen data
pollen_df = util.get_historical_pollen(
    start_date="2024-01-01",
    end_date="2025-12-01"
)

# Get historical weather data
weather_df = util.get_historical_weather(
    city=city,
    start_date="2024-01-01",
    end_date="2025-12-01",
    latitude=latitude,
    longitude=longitude
)

print(f"Pollen data shape: {pollen_df.shape}")
print(f"Weather data shape: {weather_df.shape}")

## 3. Merge and Feature Engineering

In [ ]:
# Merge datasets (both are daily data with 'date' column)
df = pd.merge(weather_df, pollen_df, on='date', how='inner')
df = df.sort_values('date').reset_index(drop=True)

# Add temporal features
df['day_of_year'] = df['date'].dt.dayofyear
df['month'] = df['date'].dt.month
df['is_high_season'] = df['day_of_year'].apply(lambda x: 1 if 140 <= x <= 250 else 0)

# Add GDD (Growing Degree Days) feature: Max(0, (T_mean - T_base))
T_base = 5.0  # Base temperature for grass growth
df['gdd_daily'] = df['temperature_2m_mean'].apply(lambda t: max(0, t - T_base))
df['gdd_cumsum'] = df.groupby(df['date'].dt.year)['gdd_daily'].cumsum()

# Add lagged weather features (yesterday's weather affects today's pollen)
df['precip_lag_1'] = df['precipitation_sum'].shift(1)
df['temp_lag_1'] = df['temperature_2m_mean'].shift(1)
df['wind_lag_1'] = df['wind_speed_10m_max'].shift(1)

df = df.dropna()
print(f"Final dataset shape: {df.shape}")
print(df.head())

## 4. Create the Feature Groups and insert the DataFrames in them

In [ ]:
fs = project.get_feature_store() 

In [ ]:
import great_expectations as ge
weather_expectation_suite = ge.core.ExpectationSuite(
    expectation_suite_name="weather_expectation_suite"
)

def expect_greater_than_zero(col):
    weather_expectation_suite.add_expectation(
        ge.core.ExpectationConfiguration(
            expectation_type="expect_column_min_to_be_between",
            kwargs={
                "column":col,
                "min_value":-0.1,
                "max_value":1000.0,
                "strict_min":True
            }
        )
    )
expect_greater_than_zero("precipitation_sum")
expect_greater_than_zero("wind_speed_10m_max")

In [ ]:
weather_fg = fs.get_or_create_feature_group(
    name='weather',
    description='Weather characteristics of each day',
    version=1,
    primary_key=['city'],
    event_time="date",
    expectation_suite=weather_expectation_suite
) 
weather_fg.insert(weather_df, wait=True)

weather_fg.update_feature_description("date", "Date of measurement of weather")
weather_fg.update_feature_description("city", "City where weather is measured/forecast for")
weather_fg.update_feature_description("temperature_2m_mean", "Temperature in Celsius")
weather_fg.update_feature_description("precipitation_sum", "Precipitation (rain/snow) in mm")
weather_fg.update_feature_description("wind_speed_10m_max", "Wind speed at 10m abouve ground")
weather_fg.update_feature_description("wind_direction_10m_dominant", "Dominant Wind direction over the dayd")

In [ ]:
fg = fs.get_feature_group("grass_pollen", version=1)
print(fg)

grass_pollen_fg = fs.get_or_create_feature_group(
    name='grass_pollen',
    description='Daily grass pollen concentration levels for Stockholm region from Pollenrapporten API',
    version=1,
    primary_key=['date'],
    event_time="date"
)

grass_pollen_fg.insert(pollen_df)
grass_pollen_fg.update_feature_description("date", "Date of pollen measurement")
grass_pollen_fg.update_feature_description("grass_pollen", "Grass pollen concentration level (0-6 scale)")
